In [6]:
import time, numpy as np, scipy.sparse as sp, cudaq_qec as qec
from cudaq_qec.plugins import nv_qldpc
from scipy.stats import binomtest

z = np.load("/kaggle/input/datasets/hummi3/gross-shots/gross_p0.004_shots.npz")
def problem(b):
    H = sp.csr_matrix((np.ones(len(z[f"{b}_H_rows"]), np.uint8), (z[f"{b}_H_rows"], z[f"{b}_H_cols"])), shape=tuple(z[f"{b}_H_shape"]))
    L = sp.csr_matrix((np.ones(len(z[f"{b}_L_rows"]), np.uint8), (z[f"{b}_L_rows"], z[f"{b}_L_cols"])), shape=tuple(z[f"{b}_L_shape"]))
    det = np.unpackbits(z[f"{b}_det"], axis=1)[:, : int(z[f"{b}_ndet"])]
    return H, L, z[f"{b}_priors"].astype(np.float64), det, z[f"{b}_obs"]

nv_failed, nv_valid, lat = {}, {}, {}
for b in "XZ":
    H, L, pri, det, obs = problem(b)
    t = time.time(); model = nv_qldpc.init_tcbp_model(H, seed=1); fit_s = time.time() - t
    dec = qec.get_decoder("nv-qldpc-decoder", H, error_rate_vec=pri, **nv_qldpc.preset("tcbp-jobs", tcbp_model=model))
    # warm-up, then single-shot latency on the first 300 shots
    dec.decode(det[0].astype(np.float64).tolist())
    ts = []
    for s in det[:300]:
        t0 = time.perf_counter(); dec.decode(s.astype(np.float64).tolist()); ts.append(time.perf_counter() - t0)
    lat[b] = np.array(ts) * 1e3
    # all shots (batched when available)
    t = time.time(); E = []
    if hasattr(dec, "decode_batch"):
        for i in range(0, len(det), 1024):
            E += [np.asarray(r.result) for r in dec.decode_batch(det[i:i+1024].astype(np.float64).tolist())]
    else:
        E = [np.asarray(dec.decode(s.astype(np.float64).tolist()).result) for s in det]
    run_s = time.time() - t
    E = (np.array(E) > 0.5).astype(np.uint8)
    nv_valid[b] = (((H @ E.T).T % 2) == det).all(axis=1)
    nv_failed[b] = (((L @ E.T).T % 2) != obs).any(axis=1) | ~nv_valid[b]
    print(f"{b}: model fit {fit_s:.1f}s ({len(model)} bytes) | decoded {len(det)} shots in {run_s:.1f}s "
          f"| valid {nv_valid[b].mean():.4f} | NVIDIA failures {nv_failed[b].sum()} "
          f"| ours {z[f'{b}_ours_failed'].sum()} | relay {z[f'{b}_relay_failed'].sum()} "
          f"| single-shot mean {lat[b].mean():.1f} ms, p99 {np.percentile(lat[b], 99):.1f} ms")

def pairs(name): return z[f"X_{name}_failed"] | z[f"Z_{name}_failed"]
nv = nv_failed["X"] | nv_failed["Z"]; ours = pairs("ours"); relay = pairs("relay")
rate = lambda f: 1 - (1 - f.mean()) ** (1 / 12)
print(f"\nfailed pairs / rate per cycle:  NVIDIA TCBP-JOBS {nv.sum()} ({rate(nv):.2e}) | ours {ours.sum()} ({rate(ours):.2e}) | relay-BP {relay.sum()} ({rate(relay):.2e})")
for name, other in [("ours", ours), ("relay-BP", relay)]:
    a, c = int((nv & ~other).sum()), int((other & ~nv).sum())
    print(f"NVIDIA vs {name}: only NVIDIA fails {a}, only {name} fails {c}, McNemar p = {binomtest(a, a + c).pvalue if a + c else 1:.3g}")
print(f"NVIDIA single-shot latency per X+Z pair (T4): mean {lat['X'].mean() + lat['Z'].mean():.1f} ms")
np.savez("/kaggle/working/nvidia_tcbpjobs_p0.004.npz", X_failed=nv_failed["X"], Z_failed=nv_failed["Z"], X_lat_ms=lat["X"], Z_lat_ms=lat["Z"])

X: model fit 141.1s (2538 bytes) | decoded 10000 shots in 100.3s | valid 1.0000 | NVIDIA failures 23 | ours 22 | relay 58 | single-shot mean 8.2 ms, p99 21.8 ms
Z: model fit 128.6s (2538 bytes) | decoded 10000 shots in 95.9s | valid 1.0000 | NVIDIA failures 18 | ours 19 | relay 45 | single-shot mean 12.8 ms, p99 27.3 ms

failed pairs / rate per cycle:  NVIDIA TCBP-JOBS 41 (3.42e-04) | ours 41 (3.42e-04) | relay-BP 103 (8.62e-04)
NVIDIA vs ours: only NVIDIA fails 4, only ours fails 4, McNemar p = 1
NVIDIA vs relay-BP: only NVIDIA fails 8, only relay-BP fails 70, McNemar p = 1.74e-13
NVIDIA single-shot latency per X+Z pair (T4): mean 21.0 ms
